# VoiceGuard - WAV Pipeline Training
THE FIX: We save HuggingFace audio as WAV files first,
then train from WAV paths using librosa.load(path) — exactly the same as inference.
This eliminates ALL domain mismatch between training and inference.

In [ ]:
!conda install -y -c conda-forge ffmpeg


In [ ]:
!pip install pandas librosa xgboost scikit-learn fsspec pyarrow huggingface_hub soundfile


In [ ]:
import pandas as pd, librosa, numpy as np, io, pickle, xgboost as xgb, warnings
import soundfile as sf
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from huggingface_hub import hf_hub_download
warnings.filterwarnings('ignore')

HF_TOKEN = 'hf_YOUR_TOKEN_HERE'

# Save audio bytes as WAV files so train/inference use IDENTICAL decoding path
os.makedirs('audio_cache/authentic', exist_ok=True)
os.makedirs('audio_cache/deepfake', exist_ok=True)

def save_audio_wav(audio_dict, out_path, sr=16000):
    try:
        y, orig_sr = librosa.load(io.BytesIO(audio_dict['bytes']), sr=sr)
        if len(y) < 1000: return False
        sf.write(out_path, y, sr)
        return True
    except: return False

for ds in [
    ('google/fleurs', 'parquet-data/hi_in/train-00000-of-00001.parquet', 'Google Fleurs', 'authentic', 1500),
    ('SPRINGLab/IndicTTS-Hindi', 'data/train-00000-of-00010.parquet', 'IndicTTS', 'authentic', 1200),
]:
    print('Downloading + saving ' + ds[2] + ' as WAV...')
    try:
        lp = hf_hub_download(repo_id=ds[0], filename=ds[1], repo_type='dataset', token=HF_TOKEN)
        df = pd.read_parquet(lp)
        count = 0
        for i, (_, row) in enumerate(tqdm(df.iterrows(), total=min(len(df), ds[4]))):
            if count >= ds[4]: break
            if 'audio' in row and row['audio'] is not None:
                out = 'audio_cache/' + ds[3] + '/' + ds[2].replace(' ','_') + '_' + str(i) + '.wav'
                if save_audio_wav(row['audio'], out): count += 1
        print('Saved ' + str(count) + ' files')
    except Exception as e: print('ERROR: ' + str(e))

for sfile in [
    'data/train-00000-of-00004.parquet',
    'data/train-00001-of-00004.parquet',
    'data/train-00002-of-00004.parquet',
    'data/train-00003-of-00004.parquet',
]:
    print('Downloading + saving Sarvam ' + sfile + ' as WAV...')
    try:
        lp = hf_hub_download(repo_id='abhirl/hindi-tts-dataset', filename=sfile, repo_type='dataset', token=HF_TOKEN)
        df = pd.read_parquet(lp)
        count = 0
        for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df))):
            if 'audio' in row and row['audio'] is not None:
                out = 'audio_cache/deepfake/sarvam_' + sfile.split('/')[-1].split('.')[0] + '_' + str(i) + '.wav'
                if save_audio_wav(row['audio'], out): count += 1
        print('Saved ' + str(count) + ' files from ' + sfile)
    except Exception as e: print('ERROR: ' + str(e))

auth_files = ['audio_cache/authentic/' + f for f in os.listdir('audio_cache/authentic') if f.endswith('.wav')]
fake_files = ['audio_cache/deepfake/' + f for f in os.listdir('audio_cache/deepfake') if f.endswith('.wav')]
print('Total authentic WAVs: ' + str(len(auth_files)))
print('Total deepfake WAVs:  ' + str(len(fake_files)))


In [ ]:
def extract_features(wav_path, sr=16000):
    y, _ = librosa.load(wav_path, sr=sr, duration=3.0)
    if len(y) < 1000: return None
    y = librosa.util.normalize(y)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    mfcc_mean  = np.mean(mfccs.T, axis=0)
    mfcc_std   = np.std(mfccs.T, axis=0)
    delta_mean = np.mean(librosa.feature.delta(mfccs).T, axis=0)[:10]
    centroid   = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr        = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff    = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    rms        = np.mean(librosa.feature.rms(y=y))
    return np.hstack((mfcc_mean, mfcc_std, delta_mean, centroid, zcr, rolloff, rms))

X, y = [], []
print('Extracting features from authentic WAV files...')
for path in tqdm(auth_files):
    f = extract_features(path)
    if f is not None: X.append(f); y.append(1)

print('Extracting features from deepfake WAV files...')
for path in tqdm(fake_files):
    f = extract_features(path)
    if f is not None: X.append(f); y.append(0)

X_final = np.array(X); y_final = np.array(y)
print('TOTAL: ' + str(len(y_final)) + ' | Auth: ' + str(int(np.sum(y_final==1))) + ' | Fake: ' + str(int(np.sum(y_final==0))))
if len(np.unique(y_final)) < 2: print('FATAL: Missing one class!')
else: print('Ready to train!')


In [ ]:
if len(np.unique(y_final)) >= 2:
    X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.15, random_state=42, stratify=y_final)
    ratio = float(np.sum(y_train==1)) / max(float(np.sum(y_train==0)), 1.0)
    print('Class balance ratio: ' + str(round(ratio,2)))
    clf = xgb.XGBClassifier(
        n_estimators=600, max_depth=9, learning_rate=0.02,
        scale_pos_weight=ratio, subsample=0.85, colsample_bytree=0.85,
        min_child_weight=3, gamma=0.1,
        n_jobs=-1, random_state=42, eval_metric='logloss'
    )
    clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)
    print('ACCURACY ON UNSEEN TEST DATA:')
    print(classification_report(y_test, clf.predict(X_test), target_names=['Deepfake','Authentic']))
    with open('voiceguard_WAV_model.pkl', 'wb') as fout: pickle.dump(clf, fout)
    print('SAVED: voiceguard_WAV_model.pkl')
    print('Download -> rename to voiceguard_rf_model.pkl -> ml_engine/models/')
